In [7]:
!pip3 install --upgrade --quiet langchain langchain-community langchain-openai chromadb
!pip3 install --upgrade --quiet pypdf pandas streamlit python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!pip install --upgrade --quiet langchain-text-splitters python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

import os
import tempfile
import streamlit as st
import pandas as pd
from dotenv import load_dotenv

C:\Users\bammi\AppData\Local\Temp\ipykernel_9876\792024501.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

In [11]:
%pip install -U langchain-google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.2
)

In [13]:
response = llm.invoke("Hello, explain RAG in one sentence.")
print(response.content)


c:\PROJECTS\rag_llms\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': '**Retrieval-Augmented Generation (RAG)** is an AI technique that enhances large language models by searching an external knowledge base for relevant facts before generating an accurate, up-to-date response.', 'extras': {'signature': 'EucTCuQTARFNMg8kSfGXsBW2rzvO6og8/TaDgs9m194pIDqyE/TjgtjFnVcVht4FtDga8kIONcvC20Gof3Tn+e0ua12nABf2BwQzJouRv4iGMxZVFPjsPi9hF6WM/AGxKF7oS2j9cAM3/dnEM6kk2Bfpi/1XT7FypYMcQAc3ehT9CT4YklxxpGdgBQFgloNCkrQceFJgDq5YGZW1Ql8XPXv2xYftuSPUj/KgTZjf5QLzeDwFeFyl8Ri2uaF+3I3wrOzwAothC+N+MVHFbHjoD1NTu/soL4ca/umNoeAnwkFya8UctKORZ3Q0VR8QozjDLvKSTJ5zi/5vRPFpblTJacZcmFVrP9YGDX2iFPnIaNhOBZl/FOYsCQ2//AXGB9cd+I6KCQ7lFuGTrhzrLcI9yngKQ+FX0ruMuIBL88o5MQ6KmT5b5Rn00jzcGfGjo23mmOugVl7/90r8PPIYoMsatjm6sqOgNkwGdZSXsXeJ5ZH7I/PHwBj97cCFq6cmXgODYLZiJ5O3dQuhufi0wejsAa+PTOs1Wrs/Lo21UP5GM+1YwMfONrj9SO8MDXnr10W7toB929FlSrl5ZyCiBqd3g05GwsEvYzCpt9WCrO6ii8gt7rHxtLHwOSlpcmie65347EX/31biB7VXYQqefD9njRhXVTOSX2q0vn93J9cZ24ISDJTHN4kYzhIMNadMqLrW0l4breMjwE+Rujpiqx2MKci8pmFPJGq0jew

Process PDF Document

Load PDF document


In [14]:
loader=PyPDFLoader("data/short-research-paper-example.pdf")
pages=loader.load()
pages

[Document(metadata={'producer': '4-Heights™ PDF Library 3.4.0.6904 (http://www.pdf-tools.com)', 'creator': '(unspecified)', 'creationdate': '2026-05-06T06:50:21+00:00', 'author': 'CollegeEssay.org', 'keywords': '', 'subject': '(unspecified)', 'title': 'Short Research Paper Example | CollegeEssay.org', 'trapped': '/False', 'moddate': '2026-05-06T07:06:30+00:00', 'source': 'data/short-research-paper-example.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Rethinking the Writing Assignment in the Age of Generative AI\n Liam Carter\n First-Year Writing Program, State University\n WRIT 101: College Composition\n Professor Hadley\n April 18, 2026\n1'),
 Document(metadata={'producer': '4-Heights™ PDF Library 3.4.0.6904 (http://www.pdf-tools.com)', 'creator': '(unspecified)', 'creationdate': '2026-05-06T06:50:21+00:00', 'author': 'CollegeEssay.org', 'keywords': '', 'subject': '(unspecified)', 'title': 'Short Research Paper Example | CollegeEssay.org', 'trapped': '/False', '

Split document

In [15]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200,length_function=len,separators=["\n\n","\n"," "])

chunks=text_splitter.split_documents(pages)

Create Embeddings

In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

def get_embedding_function():
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=GEMINI_API_KEY
    )
    return embeddings

embedding_function = get_embedding_function()

test_vector = embedding_function.embed_query("List out the reference papers")
print(test_vector)
print(len(test_vector))

[-0.014824956, -0.0076547056, 0.008995925, -0.039649058, -0.0042221844, 0.0089654075, -0.011450132, -0.009003684, -0.0027361875, 0.0031171227, 0.004867642, -0.0046014357, 0.013379424, 0.027847774, 0.09788775, 0.017879117, 0.015146028, -0.02947663, -0.0099910805, 0.0012302789, 0.032896508, 0.01338235, 0.0032128305, 0.006225647, 0.012200825, -0.0044841757, 0.014999621, 0.021031762, 0.01088151, 0.013009976, 0.021717634, -0.0052243895, -0.018487373, 0.0060593067, -0.004645517, 0.034192953, 0.008656212, -0.020737642, 0.014442286, 0.00027418128, -0.012251226, 0.0025821484, 0.0043203924, -0.00784114, 0.00011551434, -0.00477187, 0.021444289, -0.012349849, -0.022683607, 0.0283121, 0.013790546, 0.008480178, -0.008289391, -0.14009245, -0.016257476, -0.0067388755, -0.0013310281, -0.007304077, 0.006355327, 0.008772618, -0.000667742, 0.025553238, -0.0063728997, 0.009811517, -0.012215762, -0.02464228, 0.010522059, 0.022337018, -0.004767857, -0.050533444, -0.023556804, 0.010461134, 0.005968772, -0.002

In [17]:
%pip install -U langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from langchain_classic.evaluation import load_evaluator

evaluator = load_evaluator(
    evaluator="embedding_distance",
    embeddings=embedding_function
)

result = evaluator.evaluate_strings(
    prediction="Cat",
    reference="Arithmetic error"
)

print(result)

{'score': 0.2495962690295902}


Create Vector Database

In [19]:
pip install -U langchain-chroma chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import uuid
from langchain_chroma import Chroma

def create_vectorstore(chunks, embedding_function, vectorstore_path):

    # Create a list of unique ids for each document based on the content
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]

    # Ensure that only unique docs with unique ids are kept
    unique_ids = set()
    unique_chunks = []

    for chunk, id in zip(chunks, ids):
        if id not in unique_ids:
            unique_ids.add(id)
            unique_chunks.append(chunk)

    # Create a new Chroma database from the documents
    vectorstore = Chroma.from_documents(
        documents=unique_chunks,
        ids=list(unique_ids),
        embedding=embedding_function,
        persist_directory=vectorstore_path
    )

    return vectorstore


In [26]:
#create vectorstore
vectorstore=create_vectorstore(chunks=chunks,
                             embedding_function=embedding_function,
                             vectorstore_path="vectorstore_chroma")

2.Query for relevant data

In [27]:
#Load vectorstore
vectorstore = Chroma(persist_directory="vectorstore_chroma", embedding_function=embedding_function)

In [28]:
#Create retriver and get relevant chunks
retriever=vectorstore.as_retriever(search_type="similarity")
relevant_chunks=retriever.invoke("What is the title of the article?")
relevant_chunks

[Document(id='ce6b3457-50e1-5972-85ca-89b63b73c992', metadata={'subject': '(unspecified)', 'keywords': '', 'creator': '(unspecified)', 'source': 'data/short-research-paper-example.pdf', 'title': 'Short Research Paper Example | CollegeEssay.org', 'creationdate': '2026-05-06T06:50:21+00:00', 'moddate': '2026-05-06T07:06:30+00:00', 'page_label': '1', 'total_pages': 4, 'author': 'CollegeEssay.org', 'trapped': '/False', 'producer': '4-Heights™ PDF Library 3.4.0.6904 (http://www.pdf-tools.com)', 'page': 0}, page_content='Rethinking the Writing Assignment in the Age of Generative AI\n Liam Carter\n First-Year Writing Program, State University\n WRIT 101: College Composition\n Professor Hadley\n April 18, 2026\n1'),
 Document(id='2f77cfca-ae63-5a90-a733-c204af84cf3a', metadata={'total_pages': 4, 'creator': '(unspecified)', 'moddate': '2026-05-06T07:06:30+00:00', 'page_label': '2', 'page': 1, 'subject': '(unspecified)', 'trapped': '/False', 'source': 'data/short-research-paper-example.pdf', 'au

In [29]:
#Prompt template for question answering
PROMPT_TEMPLATE="""
You are an assistant for question-answering tasks.
Use the following piecces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Do not make up an answer.
{context}

Answer the question based on the above context:{question}
"""




In [30]:
#Concateneate context text

context_text="\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

#Create prompt
prompt_template=ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt=prompt_template.format_prompt(context=context_text,question="What is the title of the article?")

print(prompt)

messages=[HumanMessage(content="\nYou are an assistant for question-answering tasks.\nUse the following piecces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Do not make up an answer.\nRethinking the Writing Assignment in the Age of Generative AI\n Liam Carter\n First-Year Writing Program, State University\n WRIT 101: College Composition\n Professor Hadley\n April 18, 2026\n1\n\n---\n\nRethinking the Writing Assignment in the Age of Generative AI\n The college essay is dying, or at least changing in ways that look like dying to the\npeople most invested in its current form. Generative AI tools released since 2022 can produce\na competent five-paragraph essay in under a minute, and undergraduate students at virtually\nevery institution now have free access to these tools. The educational response has been\nuneven and often contradictory: some instructors have moved aggressively toward AI-assisted\nwriting as a pedagogical opportunity

3. Generate responses


In [31]:
llm.invoke(prompt)

c:\PROJECTS\rag_llms\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'Based on the provided context, the title of the article is **"Rethinking the Writing Assignment in the Age of Generative AI"**.', 'extras': {'signature': 'EogICoUIARFNMg+jp1OaEFJtpgOv3AmQK2OwVNUQd4gkYanW6aIVYrhy3tbDZRvuQKpp7N6fUZlaUMrfnq6vRrRFw9bwyMeGqr/Yn+uKMNnvwae7tpOzSEcXASiNUoKk5FLF2MYddlhSYwIs7aMFgGKPFMXTMQQuTvPPoMhF5NGvJ0DBNGZmlBT/BgVSqDBLSqcp5z6YWiBwxp3yX/wmz0oCgo4HseZdgwu0TzazN83AKsRL7hCf4+QEpi28OezxlL8gaFas9rrRSoMh2WO3+WP8wIkDu2eGws6Pr1+UJ/6AOlG9bP04+hJmFz7DYSDDVyfKuncb76xFiJm1nuyM6ReHSRMXFN29Wp7D11Xtxgv9QXFTQHq3FhG028dusQBVgqVh+MVEUjTivxr0d8crZPiQ6rq6kj+HvpKctLvILXaZAQPK3eO3g9yeaxkixxorpn8SLgUV4JDYOUV2O3AbvqRfpZ+lJK6MygteyJdxfp31YbishGMxRf3hR13e7pDIKMgTuW/2bGSGeP6PnHukmi5/n3GBm7CqI3VO3FrLbCyWz/sTq33llU+o40Q0f10C+Bly/TdQ+1Kah+hGAUAJw0XHX2ZKrKiyalMuNib5rsTBjJOtLDaEUtNktnNK+o5rM1sV0cwE1bf/ZUcRI/LzfCsb4Q+dFbJ/fWrqOopPr5fmk4nknp2xFV6qLbi6Vuuc/fAJu2nch5aiCbp9rilS84uSDPHTCvNwvwVqu+yXRS6kyijng7fmnGBX5xNWBkKhbEZSc7GQF50wm7W8+JGASBRcBOQCZ2VN

Using Langchain Expression Language

In [32]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain=(
    {"context":retriever|format_docs,"question":RunnablePassthrough()}
    |prompt_template
    |llm
)
rag_chain.invoke("What is the title of the article?")

c:\PROJECTS\rag_llms\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'Based on the context provided, the title of the article is **"Rethinking the Writing Assignment in the Age of Generative AI"**.', 'extras': {'signature': 'Er4HCrsHARFNMg+NUcK6S6l2S5JPO+mq9E9OZs0atTl+ayo6gM7WrGvl3m0M58ZAxvdeWCGbVFQXc8oiH2dsUJEu76tbcK+Y6Sf4+J0S9ruYrBFnuqNjL0IJsuJQG6KSFikBFynCgFtaPzilP9JvUiL7uV9srebroHUYrc6+bYuZypvo/QHZOvF8N5+mv+0066IVVTt4JsNa0bK1gTl6h7tC8k21kbpSH4ghhcFdMLa+yqmGjuIWRWM8/qCuYkV3wN6v8NXuy5bdfjIoZhb7USU4Y8r9X/nhxcVO+xvFfTF7Ags0Co8vcJyJJyLRLNlmrHyDKeAYkU2wcBvC0HH/hnM0euHl1bF5xvGwVZcA+coJq0ANBGuS4lyxFy/mOteq8d6q20ePmyEv4lsqYuFp+M08cYKZkx2HPRlbMoEMpFWTxIYzHR5U6g4AiyjfKCQWH+4DHYdff3uefnneTV6ei7MnwE9HOCAiLgCAxhNffAlptDzCXVnUH83JnE4RwRuN/VsVbmAIWWvbRo2HubwVxh0YuNcBcK+/irQDC2Ndk4kDlmh6G0/oBPPkqR6I+2oG8tfOrQAARdK+NJV+d7HE5h8Fsqzo3dVlkrH+JX54x7ftgqw5HWgrcvx/7c6pX/Vq0Hvdx14dGI20wyLXfP1Ba3TlAJaJqIIq3th9sAQViMysFFmfJKmjxSXC4j1JJYOusQpt6EbXoCrcKOj784tT4QG59B96sW0EIBOIoZ9p4lwbx2OP/AkCg7ujK91kee3bd91IMctK1/6mO/hzuZPcCRwgHu2fmecY

Generating Structured Output

In [41]:
class AnswerwithSources(BaseModel):
    """Answer with sources"""
    answer:str=Field(description="Answer to the question")
    sources:list[str]=Field(description="List of sources used to answer the question")
    reasoning:str=Field(description="Reasoning behind the answer")

class ExtractedInfo(BaseModel):
    """Extracted information about the article"""
    paper_title:AnswerwithSources
    paper_summary:AnswerwithSources
    paper_authors:AnswerwithSources
    paper_publication_date:AnswerwithSources

In [42]:
rag_chain=(
    {"context":retriever|format_docs,"question":RunnablePassthrough()}
    |prompt_template
    |llm.with_structured_output(ExtractedInfo, strict=True)
)

rag_chain.invoke("Give me the title,summary,publication date and authors of the research paper?")

c:\PROJECTS\rag_llms\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ExtractedInfo(paper_title=AnswerwithSources(answer='Rethinking the Writing Assignment in the Age of Generative AI', sources=['Rethinking the Writing Assignment in the Age of Generative AI'], reasoning='The title is explicitly written at the top of the header and title section of the text.'), paper_summary=AnswerwithSources(answer="The paper discusses the impact of generative AI on the college essay, noting that blanket prohibition or full embrace are both inadequate responses. It argues that writing assignments should be redesigned around cognitive work AI cannot easily replace, such as developing an argument over time, integrating a writer's own reading history, and documenting the drafting process.", sources=['Generative AI tools released since 2022 can produce a competent five-paragraph essay in under a minute, and undergraduate students at virtually every institution now have free access to these tools.', "This paper argues that the question will not resolve itself, and that the ri

Transform Response into a dataframe


In [47]:
import pandas as pd

# Invoke RAG chain
structured_response = rag_chain.invoke(
    "Give me the title, summary, publication date, and authors of the research paper?"
)

# Inspect the response
print("TYPE:", type(structured_response))
print("RESPONSE:")
print(structured_response)

c:\PROJECTS\rag_llms\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


TYPE: <class '__main__.ExtractedInfo'>
RESPONSE:
paper_title=AnswerwithSources(answer='Rethinking the Writing Assignment in the Age of Generative AI', sources=['Rethinking the Writing Assignment in the Age of Generative AI'], reasoning='The title is clearly stated at the top of the document header and title page.') paper_summary=AnswerwithSources(answer='The paper discusses the challenge generative AI poses to traditional college composition assignments. It argues against both total prohibition and full embrace of AI, advocating instead for redesigning writing assignments around cognitive tasks that AI cannot easily replace, such as developing arguments over time, integrating personal reading history, and documenting the drafting process.', sources=["This paper argues that the question will not resolve itself, and that the right educational response is neither full embrace nor full prohibition. It is a redesign of the writing assignment around the cognitive work that AI cannot easily r

In [50]:
import pandas as pd

# Invoke RAG chain
structured_response = rag_chain.invoke(
    "Give me the title, summary, publication date, and authors of the research paper?"
)

# Create rows
answer_row = [
    structured_response.paper_title.answer,
    structured_response.paper_summary.answer,
    structured_response.paper_publication_date.answer,
    structured_response.paper_authors.answer
]

source_row = [
    structured_response.paper_title.sources,
    structured_response.paper_summary.sources,
    structured_response.paper_publication_date.sources,
    structured_response.paper_authors.sources
]

reasoning_row = [
    structured_response.paper_title.reasoning,
    structured_response.paper_summary.reasoning,
    structured_response.paper_publication_date.reasoning,
    structured_response.paper_authors.reasoning
]

# Create final table
structured_response_df = pd.DataFrame(
    [answer_row, source_row, reasoning_row],
    columns=["Title", "Summary", "Publication Date", "Authors"],
    index=["Answer", "Sources", "Reasoning"]
)

structured_response_df

c:\PROJECTS\rag_llms\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,Title,Summary,Publication Date,Authors
Answer,Rethinking the Writing Assignment in the Age o...,The paper addresses the impact of generative A...,"April 18, 2026",Liam Carter
Sources,[Rethinking the Writing Assignment in the Age ...,[Generative AI tools released since 2022 can p...,"[April 18, 2026]",[Liam Carter]
Reasoning,The title is prominently listed at the beginni...,The text details how generative AI produces co...,"The date April 18, 2026 is explicitly listed o...",Liam Carter is specified as the author directl...


In [49]:
print(structured_response.model_fields.keys())


dict_keys(['paper_title', 'paper_summary', 'paper_authors', 'paper_publication_date'])


C:\Users\bammi\AppData\Local\Temp\ipykernel_9876\2047898558.py:1: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print(structured_response.model_fields.keys())
